In [1]:
# Notebook 1 — Data audit and schema alignment
import os
import re
import hashlib
import random
import numpy as np
import pandas as pd
from datetime import datetime

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CANONICAL_COLS = [
    "id", "osm_id", "osm_geom_type",
    "osm_tag_key", "osm_tag_value",
    "name", "name_ur", "alt_name",
    "description_raw", "description_final",
    "description_source",
    "wikipedia_title", "wikipedia_url",
    "lat", "lon", "city", "province", "country",
    "language", "dedup_group_id"
]

TIMESTAMP = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")
print("Seed set. Canonical columns defined.")

Seed set. Canonical columns defined.


C:\Users\stran\AppData\Local\Temp\ipykernel_3672\1616994383.py:25: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  TIMESTAMP = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")


In [2]:
#Utility: Urdu diacritic stripping, normalization, language detection (English/Urdu/Roman Urdu/Mixed)

In [3]:
import unicodedata

URDU_DIACRITICS = set([
    "\u064b", "\u064c", "\u064d", "\u064e", "\u064f", "\u0650", "\u0651", "\u0652",
    "\u0653", "\u0654", "\u0655", "\u0670"
])

def strip_urdu_diacritics(text: str) -> str:
    if not isinstance(text, str):
        return text
    return "".join(ch for ch in text if ch not in URDU_DIACRITICS)

def normalize_text(s: str) -> str:
    if not isinstance(s, str) or not s:
        return ""
    s = s.replace("\u200c", " ")  # zero-width non-joiner to space
    s = strip_urdu_diacritics(s)
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Simple heuristics: Urdu script detection, Roman Urdu cues, English letters
def detect_language_heuristic(s: str) -> str:
    if not isinstance(s, str) or not s.strip():
        return "Unknown"
    has_urdu_script = any("\u0600" <= ch <= "\u06FF" for ch in s)  # Arabic/Urdu block
    has_latin = any("A" <= ch <= "Z" or "a" <= ch <= "z" for ch in s)
    roman_urdu_cues = ["hai", "kia", "kya", "bazar", "masjid", "chowk", "ghar", "khana", "bache", "park", "sadak", "bazaar"]
    has_roman_urdu = any(tok in s.lower() for tok in roman_urdu_cues)

    if has_urdu_script and not has_latin:
        return "Urdu"
    if has_latin and not has_urdu_script:
        return "Roman Urdu" if has_roman_urdu else "English"
    if has_latin and has_urdu_script:
        return "Mixed"
    return "Unknown"

In [4]:
import unicodedata

URDU_DIACRITICS = set([
    "\u064b", "\u064c", "\u064d", "\u064e", "\u064f", "\u0650", "\u0651", "\u0652",
    "\u0653", "\u0654", "\u0655", "\u0670"
])

def strip_urdu_diacritics(text: str) -> str:
    if not isinstance(text, str):
        return text
    return "".join(ch for ch in text if ch not in URDU_DIACRITICS)

def normalize_text(s: str) -> str:
    if not isinstance(s, str) or not s:
        return ""
    s = s.replace("\u200c", " ")  # zero-width non-joiner to space
    s = strip_urdu_diacritics(s)
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Simple heuristics: Urdu script detection, Roman Urdu cues, English letters
def detect_language_heuristic(s: str) -> str:
    if not isinstance(s, str) or not s.strip():
        return "Unknown"
    has_urdu_script = any("\u0600" <= ch <= "\u06FF" for ch in s)  # Arabic/Urdu block
    has_latin = any("A" <= ch <= "Z" or "a" <= ch <= "z" for ch in s)
    roman_urdu_cues = ["hai", "kia", "kya", "bazar", "masjid", "chowk", "ghar", "khana", "bache", "park", "sadak", "bazaar"]
    has_roman_urdu = any(tok in s.lower() for tok in roman_urdu_cues)

    if has_urdu_script and not has_latin:
        return "Urdu"
    if has_latin and not has_urdu_script:
        return "Roman Urdu" if has_roman_urdu else "English"
    if has_latin and has_urdu_script:
        return "Mixed"
    return "Unknown"

In [5]:
raw_dir = "data/raw"
candidate_files = []
if os.path.isdir(raw_dir):
    for fn in os.listdir(raw_dir):
        if fn.lower().endswith((".csv", ".parquet")):
            candidate_files.append(os.path.join(raw_dir, fn))

df_raw_list = []
for path in candidate_files:
    try:
        if path.endswith(".csv"):
            df_raw_list.append(pd.read_csv(path))
        elif path.endswith(".parquet"):
            df_raw_list.append(pd.read_parquet(path))
    except Exception as e:
        print("Skipping unreadable file:", path, "->", e)

if df_raw_list:
    df_raw = pd.concat(df_raw_list, ignore_index=True)
    source_note = f"Loaded {len(candidate_files)} raw file(s) from data/raw"
else:
    # Toy sample reflecting Islamabad context
    df_raw = pd.DataFrame([
        {
            "id": "toy-1",
            "osm_id": "n1",
            "osm_geom_type": "node",
            "osm_tag_key": "amenity",
            "osm_tag_value": "park",
            "name": "Kids Park F-7",
            "name_ur": "بچوں کا پارک ایف-7",
            "alt_name": "",
            "description_raw": "A quiet park near the F-7 markaz, perfect for kids.",
            "description_source": "toy",
            "wikipedia_title": "",
            "wikipedia_url": "",
            "lat": 33.723, "lon": 73.055,
            "city": "Islamabad", "province": "ICT", "country": "Pakistan",
        },
        {
            "id": "toy-2",
            "osm_id": "w2",
            "osm_geom_type": "way",
            "osm_tag_key": "amenity",
            "osm_tag_value": "place_of_worship",
            "name": "Masjid-e-Quba",
            "name_ur": "مسجد قبا",
            "alt_name": "Quba Mosque",
            "description_raw": "پرانے بازار کے قریب ایک خوبصورت مسجد۔",
            "description_source": "toy",
            "wikipedia_title": "",
            "wikipedia_url": "",
            "lat": 33.706, "lon": 73.039,
            "city": "Islamabad", "province": "ICT", "country": "Pakistan",
        },
        {
            "id": "toy-3",
            "osm_id": "n3",
            "osm_geom_type": "node",
            "osm_tag_key": "shop",
            "osm_tag_value": "mall",
            "name": "Centaurus",
            "name_ur": "سینٹورس",
            "alt_name": "The Centaurus Mall",
            "description_raw": "Famous mall with food court and cinema.",
            "description_source": "toy",
            "wikipedia_title": "The Centaurus",
            "wikipedia_url": "",
            "lat": 33.710, "lon": 73.058,
            "city": "Islamabad", "province": "ICT", "country": "Pakistan",
        }
    ])
    source_note = "Created toy sample (no raw files found)."

print(source_note)
print("Raw shape:", df_raw.shape)
df_raw.head()

Created toy sample (no raw files found).
Raw shape: (3, 17)


,id,osm_id,osm_geom_type,osm_tag_key,osm_tag_value,name,name_ur,alt_name,description_raw,description_source,wikipedia_title,wikipedia_url,lat,lon,city,province,country
0,toy-1,n1,node,amenity,park,Kids Park F-7,بچوں کا پارک ایف-7,,"A quiet park near the F-7 markaz, perfect for ...",toy,,,33.723,73.055,Islamabad,ICT,Pakistan
1,toy-2,w2,way,amenity,place_of_worship,Masjid-e-Quba,مسجد قبا,Quba Mosque,پرانے بازار کے قریب ایک خوبصورت مسجد۔,toy,,,33.706,73.039,Islamabad,ICT,Pakistan
2,toy-3,n3,node,shop,mall,Centaurus,سینٹورس,The Centaurus Mall,Famous mall with food court and cinema.,toy,The Centaurus,,33.710,73.058,Islamabad,ICT,Pakistan


In [6]:
df = df_raw.copy()

# Ensure canonical columns exist; add if missing
for col in CANONICAL_COLS:
    if col not in df.columns:
        df[col] = pd.Series([None] * len(df))

# Prefer description_raw if description_final missing; we'll normalize in next cell
df["description_final"] = df["description_final"].fillna(df["description_raw"])

# Basic types
for num_col in ["lat", "lon"]:
    df[num_col] = pd.to_numeric(df[num_col], errors="coerce")

# Order columns
df = df[CANONICAL_COLS].copy()
print("Aligned to canonical schema with", len(CANONICAL_COLS), "columns.")
df.head()

Aligned to canonical schema with 20 columns.


,id,osm_id,osm_geom_type,osm_tag_key,osm_tag_value,name,name_ur,alt_name,description_raw,description_final,description_source,wikipedia_title,wikipedia_url,lat,lon,city,province,country,language,dedup_group_id
0,toy-1,n1,node,amenity,park,Kids Park F-7,بچوں کا پارک ایف-7,,"A quiet park near the F-7 markaz, perfect for ...","A quiet park near the F-7 markaz, perfect for ...",toy,,,33.723,73.055,Islamabad,ICT,Pakistan,None,None
1,toy-2,w2,way,amenity,place_of_worship,Masjid-e-Quba,مسجد قبا,Quba Mosque,پرانے بازار کے قریب ایک خوبصورت مسجد۔,پرانے بازار کے قریب ایک خوبصورت مسجد۔,toy,,,33.706,73.039,Islamabad,ICT,Pakistan,None,None
2,toy-3,n3,node,shop,mall,Centaurus,سینٹورس,The Centaurus Mall,Famous mall with food court and cinema.,Famous mall with food court and cinema.,toy,The Centaurus,,33.710,73.058,Islamabad,ICT,Pakistan,None,None


In [7]:
df.columns

Index(['id', 'osm_id', 'osm_geom_type', 'osm_tag_key', 'osm_tag_value', 'name',
       'name_ur', 'alt_name', 'description_raw', 'description_final',
       'description_source', 'wikipedia_title', 'wikipedia_url', 'lat', 'lon',
       'city', 'province', 'country', 'language', 'dedup_group_id'],
      dtype='object')

In [8]:
text_cols = ["name", "name_ur", "alt_name", "description_raw", "description_final"]

for c in text_cols:
    df[c] = df[c].apply(normalize_text)

# Language detection primarily on description_final; fallback to name
df["language"] = df["description_final"].apply(detect_language_heuristic)
mask_unknown = df["language"].isin(["Unknown", None, ""])
df.loc[mask_unknown, "language"] = df.loc[mask_unknown, "name"].apply(detect_language_heuristic)

print("Language distribution:")
print(df["language"].value_counts(dropna=False))
df.head()

Language distribution:
language
Roman Urdu    1
Urdu          1
English       1
Name: count, dtype: int64


,id,osm_id,osm_geom_type,osm_tag_key,osm_tag_value,name,name_ur,alt_name,description_raw,description_final,description_source,wikipedia_title,wikipedia_url,lat,lon,city,province,country,language,dedup_group_id
0,toy-1,n1,node,amenity,park,Kids Park F-7,بچوں کا پارک ایف-7,,"A quiet park near the F-7 markaz, perfect for ...","A quiet park near the F-7 markaz, perfect for ...",toy,,,33.723,73.055,Islamabad,ICT,Pakistan,Roman Urdu,None
1,toy-2,w2,way,amenity,place_of_worship,Masjid-e-Quba,مسجد قبا,Quba Mosque,پرانے بازار کے قریب ایک خوبصورت مسجد۔,پرانے بازار کے قریب ایک خوبصورت مسجد۔,toy,,,33.706,73.039,Islamabad,ICT,Pakistan,Urdu,None
2,toy-3,n3,node,shop,mall,Centaurus,سینٹورس,The Centaurus Mall,Famous mall with food court and cinema.,Famous mall with food court and cinema.,toy,The Centaurus,,33.710,73.058,Islamabad,ICT,Pakistan,English,None


In [9]:
def md5(s: str) -> str:
    if not isinstance(s, str):
        s = str(s)
    return hashlib.md5(s.encode("utf-8")).hexdigest()

# Exact duplicate groups by normalized description_final
df["exact_sig"] = df["description_final"].fillna("").str.lower().apply(md5)

# Near-duplicate: lowercased name within city
df["near_sig"] = (df["city"].fillna("").str.lower() + "||" + df["name"].fillna("").str.lower()).apply(md5)

# Dedup group id: combine exact and near sigs
df["dedup_group_id"] = (df["exact_sig"].str[:8] + "-" + df["near_sig"].str[:8])

# Choose representative row per group: first occurrence
df["rep_rank"] = df.groupby("dedup_group_id").cumcount()
df_rep = df[df["rep_rank"] == 0].copy()

print("Rows before:", len(df), " | after dedup (representatives):", len(df_rep))
df_rep = df_rep.drop(columns=["exact_sig", "near_sig", "rep_rank"])
df_rep.head()

Rows before: 3  | after dedup (representatives): 3


,id,osm_id,osm_geom_type,osm_tag_key,osm_tag_value,name,name_ur,alt_name,description_raw,description_final,description_source,wikipedia_title,wikipedia_url,lat,lon,city,province,country,language,dedup_group_id
0,toy-1,n1,node,amenity,park,Kids Park F-7,بچوں کا پارک ایف-7,,"A quiet park near the F-7 markaz, perfect for ...","A quiet park near the F-7 markaz, perfect for ...",toy,,,33.723,73.055,Islamabad,ICT,Pakistan,Roman Urdu,0dc2b872-da697349
1,toy-2,w2,way,amenity,place_of_worship,Masjid-e-Quba,مسجد قبا,Quba Mosque,پرانے بازار کے قریب ایک خوبصورت مسجد۔,پرانے بازار کے قریب ایک خوبصورت مسجد۔,toy,,,33.706,73.039,Islamabad,ICT,Pakistan,Urdu,05f94746-e0e4091e
2,toy-3,n3,node,shop,mall,Centaurus,سینٹورس,The Centaurus Mall,Famous mall with food court and cinema.,Famous mall with food court and cinema.,toy,The Centaurus,,33.710,73.058,Islamabad,ICT,Pakistan,English,b6ece0a2-82d7bdcf
